# The $(\alpha, K)$ difficulty grid

Two questions on one grid of mixtures, using the same draws for both.

**Is the separation condition plausible?** Assumption 3 asks that
$\eta = \gamma_{\mathrm{out}} - \gamma_{\mathrm{in}}$ be positive. At a finite horizon we
observe $\eta_n$, not $\eta$, and only through an estimate. Each mixture is therefore
classified from *simultaneous* confidence bounds on the entries of $\Gamma^{(n)}$:
**separated** when the lower bound is positive, **nonseparated** when the upper bound is
negative, **uncertain** in between. The third class is what the published Figure 2 could
not express: it estimated each $\Gamma_{k\ell}$ from two realisations at a single horizon
and read off the sign of a point estimate, so cells near the boundary reported the sign of
their own noise.

**Does clustering recover the partition, and does the data-driven rule find $K$?** Exact
recovery is the primary outcome, being what Theorems 3.5, 3.8 and 3.9 are statements about;
ARI is kept as a graded second reading of the same partition.

$\Gamma^{(n)}$ is estimated on an *independent* Monte Carlo sample, never on the
dissimilarity matrix the clustering ran on. Kernels are drawn once per `mixture_id` and held
fixed, so all four algorithms and both horizons see the same mixture and the comparison
between them is paired.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from experiments import wilson_interval
from figures import DIVERGING_CMAP, PAPER_STYLE, SEQUENTIAL_CMAP

## Setup

The sweeps themselves live in `sweep_recovery.py` and `sweep_khat.py`, which write tidy
tables. Recomputing them takes hours; the figures below read the tables. Set `RECOMPUTE` to
regenerate, or run the scripts from a shell.

In [ ]:
RECOMPUTE = False        # True re-runs the sweeps: about 2 h each on 32 cores

RESULTS   = Path("results")
FIGURES   = Path("Figures/Grid")
HORIZON   = 1000         # the horizon the figures report; the sweeps also store n = 400
ALPHAS    = [0.1, 0.2, 0.3, 0.4, 0.5, 1.0, 5.0, 10.0]
KS        = list(range(2, 11))
ALGOS      = ["average", "pam"]      # the main comparison
ALGOS_ALL  = ["single", "complete", "average", "pam"]   # everything the sweep scored
LEVEL     = 0.95

FIGURES.mkdir(parents=True, exist_ok=True)

if RECOMPUTE:
    import subprocess
    subprocess.run(["python3", "sweep_recovery.py", "--tag", "main"], check=True)
    subprocess.run(["python3", "sweep_khat.py", "--asw", "--tag", "final",
                    "--horizons", "1000"], check=True)

cluster = pd.read_csv(RESULTS / "recovery_cluster_main.csv")
eta     = pd.read_csv(RESULTS / "recovery_eta_main.csv")
khat    = pd.read_csv(RESULTS / "khat_grid_final.csv")

cluster = cluster[cluster.n == HORIZON]
eta     = eta[eta.n == HORIZON]
khat    = khat[khat.n == HORIZON]

R_MIX = int(eta.groupby(["alpha", "K"]).mixture_id.nunique().max())
print(f"{len(ALPHAS)} x {len(KS)} cells, {R_MIX} mixtures each, N = {int(cluster.N.iloc[0])}, "
      f"n = {HORIZON}")

## Reading a cell

Every number below is a Monte Carlo proportion over `R_MIX` mixtures, so every number
carries a Wilson interval. `grid` turns any per-mixture indicator into the array the
heatmaps draw, and `frame` is the axis furniture of the published figures.

In [ ]:
def grid(df, value, agg="mean"):
    """(len(ALPHAS), len(KS)) array of `value` aggregated per cell."""
    table = df.pivot_table(index="alpha", columns="K", values=value, aggfunc=agg)
    return table.reindex(index=ALPHAS, columns=KS).to_numpy(dtype=float)


def wilson_half_width(df, value):
    """Half-width of the Wilson interval on each cell's proportion, for the annotations."""
    out = np.full((len(ALPHAS), len(KS)), np.nan)
    for i, a in enumerate(ALPHAS):
        for j, K in enumerate(KS):
            s = df[(df.alpha == a) & (df.K == K)][value]
            if len(s):
                lo, hi = wilson_interval(int(s.sum()), len(s), LEVEL)
                out[i, j] = (hi - lo) / 2
    return out


def frame(ax, title):
    ax.set_xticks(range(len(KS)), [str(K) for K in KS])
    ax.set_yticks(range(len(ALPHAS)), [f"{a:g}" for a in ALPHAS])
    ax.set_xlabel(r"$K$")
    ax.set_ylabel(r"$\alpha$")
    ax.set_title(title, fontsize=9, pad=6)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.4)
        spine.set_color("0.7")


def heatmap(ax, P, annot=None, title="", cmap=SEQUENTIAL_CMAP, fmt="{:.2f}"):
    """Colour = the proportion; annotation = that proportion and, below it, `annot`."""
    im = ax.imshow(P, origin="lower", aspect="auto", cmap=cmap, vmin=0.0, vmax=1.0)
    frame(ax, title)
    for i in range(P.shape[0]):
        for j in range(P.shape[1]):
            if not np.isfinite(P[i, j]):
                continue
            col = "white" if P[i, j] > 0.55 else "0.25"
            ax.text(j, i + 0.13, fmt.format(P[i, j]), ha="center", va="center",
                    fontsize=5.5, color=col)
            if annot is not None and np.isfinite(annot[i, j]):
                ax.text(j, i - 0.17, f"{annot[i, j]:+.2f}", ha="center", va="center",
                        fontsize=4.6, color=col, alpha=0.85)
    return im


def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIGURES / f"{name}.{ext}", bbox_inches="tight")
    print("figure written to", FIGURES / f"{name}.pdf")

## Plausibility of the separation condition

Colour is the **verdict balance**, $\Pr(\text{separated}) - \Pr(\text{nonseparated})$ at
level 0.95: blue where the interval establishes $\eta_n > 0$, red where it establishes
$\eta_n < 0$, and pale wherever it cannot decide. A pale cell is not a cell where the
condition fails; it is one where the horizon is too short to tell, and the published
Figure 2 had no way of saying so -- it read the sign of a point estimate and coloured the
cell as though the answer were known.

The big number is the separated share, the small one the median $\hat\eta_n$, so a cell at
0 still says how far it is.

In [ ]:
eta = eta.assign(
    is_separated=(eta.separation_status == "separated").astype(int),
    is_uncertain=(eta.separation_status == "uncertain").astype(int),
    is_nonseparated=(eta.separation_status == "nonseparated").astype(int),
)

P_sep = grid(eta, "is_separated")
P_non = grid(eta, "is_nonseparated")
balance = P_sep - P_non                 # +1 all separated, -1 all not, 0 no verdict
margin = grid(eta, "eta_hat", agg="median")

with plt.rc_context(PAPER_STYLE):
    fig, ax = plt.subplots(figsize=(5.4, 4.0))
    im = ax.imshow(balance, origin="lower", aspect="auto", cmap=DIVERGING_CMAP,
                   vmin=-1.0, vmax=1.0)
    frame(ax, r"Assumption 3 at $n = %d$" % HORIZON)
    for i in range(balance.shape[0]):
        for j in range(balance.shape[1]):
            if not np.isfinite(balance[i, j]):
                continue
            col = "white" if abs(balance[i, j]) > 0.6 else "0.25"
            ax.text(j, i + 0.13, f"{P_sep[i, j]:.2f}", ha="center", va="center",
                    fontsize=5.5, color=col)
            ax.text(j, i - 0.17, f"{margin[i, j]:+.2f}", ha="center", va="center",
                    fontsize=4.6, color=col, alpha=0.85)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03,
                      ticks=[-1, -0.5, 0, 0.5, 1])
    cb.set_label(r"$\Pr(\eta_n > 0$ established$) - \Pr(\eta_n < 0$ established$)$",
                 fontsize=8)
    cb.ax.set_yticklabels(["all\nnot separated", "", "no\nverdict", "", "all\nseparated"],
                          fontsize=6)
    cb.outline.set_visible(False)
    fig.tight_layout()
    save(fig, "separation")
    plt.show()

print("median Wilson half-width on the separated proportion: "
      f"{np.nanmedian(wilson_half_width(eta, 'is_separated')):.3f}")

## Exact recovery at known $K$

Average linkage and PAM. Complete linkage is dropped: paired over the grid it is
indistinguishable from average (79 wins against 107, $p = 0.05$), so reporting both says
nothing. Single linkage is the estimator Theorem 3.5 is actually about, and is kept in the
appendix section below, where the published Figures 9 and 10 had it.

The linkages are cut at exactly the first $N-K$ merges rather than by a height threshold,
so every one of them obeys literally the same rule; PAM is the strictly-improving one-swap
algorithm of Theorem 3.8, and every run carries the flag certifying that its medoid set is
one-swap stationary.

In [ ]:
with plt.rc_context(PAPER_STYLE):
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.0), sharey=True)
    for ax, algo in zip(axes.ravel(), ALGOS):
        sub = cluster[cluster.algorithm == algo]
        im = heatmap(ax, grid(sub, "exact_recovery"), grid(sub, "ari"), algo)
    cb = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02)
    cb.set_label("probability of exact recovery (annotated: mean ARI)", fontsize=8)
    cb.outline.set_visible(False)
    save(fig, "recovery")
    plt.show()

certified = cluster[cluster.algorithm == "pam"]
print(f"PAM certified one-swap stationary in "
      f"{int(certified.pam_one_swap_certified.sum())}/{len(certified)} runs; "
      f"hit the swap cap {int(certified.pam_hit_cap.sum())} times")

## The chain the section rests on

$(\alpha, K)$ is a *generative* difficulty knob, not the separation condition itself. What
governs recovery is $\eta_n$, and the grid is only a way of sweeping through it. Binning the
same mixtures by their estimated $\eta_n$ shows the chain directly.

In [ ]:
merged = cluster.merge(
    eta[["alpha", "K", "mixture_id", "eta_hat", "separation_status"]],
    on=["alpha", "K", "mixture_id"], how="left")

edges  = [-np.inf, 0.0, 0.02, 0.05, 0.10, 0.20, np.inf]
labels = [r"$\eta_n<0$", "0–.02", ".02–.05", ".05–.10", ".10–.20", r"$>$.20"]
merged["band"] = pd.cut(merged.eta_hat, edges, labels=labels)

rows = []
for band in labels:
    s = merged[merged.band == band]
    if not len(s):
        continue
    row = {"eta band": band, "mixtures": len(s) // len(ALGOS_ALL)}
    for a in ALGOS_ALL:
        row[a] = f"{100 * s[s.algorithm == a].exact_recovery.mean():.0f}%"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

with plt.rc_context(PAPER_STYLE):
    fig, ax = plt.subplots(figsize=(5.2, 3.4))
    for algo in ALGOS_ALL:
        s = merged[merged.algorithm == algo]
        rate = [s[s.band == b].exact_recovery.mean() for b in labels]
        ax.plot(range(len(labels)), rate, marker="o", ms=4, label=algo)
    ax.set_xticks(range(len(labels)), labels)
    ax.set_xlabel(r"estimated $\eta_n$")
    ax.set_ylabel("probability of exact recovery")
    ax.set_ylim(-0.03, 1.03)
    ax.legend(loc="lower right")
    fig.tight_layout()
    save(fig, "recovery_vs_eta")
    plt.show()

## Selecting $K$ from the data

Three rules on the same matrices. `stated` is the threshold of the current Theorem 3.9,
$M(\log N/n)^{1/4}$; `safeguard` is the proposed replacement,
$\max\{\sqrt{h_{\mathrm{med}}h_{\max}},\ h_{\max}(\log N/n)^{1/4}\}$ over the single-linkage
merge heights of the profile distances; `asw-pam` is what applied sequence analysis does,
maximising the average silhouette width over a range of $k$.

Reported separately, as the two questions they are: does the rule find $K$, and does the
partition it implies recover $\mathcal P^\star$.

In [ ]:
RULES = ["safeguard", "asw-pam", "stated"]
TITLES = {"safeguard": r"safeguarded profile rule",
          "asw-pam":   r"maximal silhouette width (PAM)",
          "stated":    r"threshold of Theorem 3.9, $M(\log N/n)^{1/4}$"}

with plt.rc_context(PAPER_STYLE):
    fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.6), sharey=True)
    for ax, rule in zip(axes, RULES):
        sub = khat[khat.rule == rule]
        im = heatmap(ax, grid(sub, "k_correct"), grid(sub, "exact_recovery"), TITLES[rule])
    cb = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)
    cb.set_label(r"$\Pr(\hat K = K)$ (annotated: $\Pr$ exact partition)", fontsize=8)
    cb.outline.set_visible(False)
    save(fig, "k_selection")
    plt.show()

summary = []
for rule in RULES:
    s = khat[khat.rule == rule]
    lo, hi = wilson_interval(int(s.k_correct.sum()), len(s), LEVEL)
    summary.append({"rule": rule, "K_hat = K": f"{100 * s.k_correct.mean():.1f}%",
                    "95% CI": f"[{100*lo:.0f}, {100*hi:.0f}]",
                    "exact partition": f"{100 * s.exact_recovery.mean():.1f}%"})
print(pd.DataFrame(summary).to_string(index=False))

## The price of not knowing $K$

Cutting at $\hat K$ rather than at the true $K$ costs the difference between the two columns
below. It is not the same as the error rate of the rule: a wrong $\hat K$ can still leave
most of the partition intact, and a right $\hat K$ does not guarantee the partition.

In [ ]:
rows = []
for algo in ALGOS_ALL:
    s = cluster[cluster.algorithm == algo]
    rows.append({"algorithm": algo,
                 "known K": f"{100 * s.exact_recovery.mean():.1f}%",
                 "at K_hat": f"{100 * s.exact_recovery_at_k_hat.mean():.1f}%",
                 "K_hat correct": f"{100 * s.k_correct.mean():.1f}%"})
print(pd.DataFrame(rows).to_string(index=False))

## Appendix: single linkage

The estimator Theorem 3.5 is about. It is the weakest of the three in practice -- single
linkage merges two blocks on the *smallest* dissimilarity between them, so one aberrant
sequence chains them together and the cut at $K$ then spends a whole block on that sequence.
Average linkage carries the main figure for that reason; the theory covers both, and every
bracketed linkage between them, by Remark 3.4.

In [ ]:
with plt.rc_context(PAPER_STYLE):
    fig, ax = plt.subplots(figsize=(5.4, 4.0))
    sub = cluster[cluster.algorithm == "single"]
    im = heatmap(ax, grid(sub, "exact_recovery"), grid(sub, "ari"), "single linkage")
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
    cb.set_label("probability of exact recovery (annotated: mean ARI)", fontsize=8)
    cb.outline.set_visible(False)
    fig.tight_layout()
    save(fig, "recovery_single_linkage")
    plt.show()